# Block 5 — Addressee Prediction on Validation Set

Runs LLM addressee tagging on `data_for_prediction.xlsx` (270 rows, 15 per film across 18 films).

Each row already contains `scene_text` — the full scene dialogue used as context.
The LLM predicts `addressee_llm` and `addressee_type_llm` for the **target utterance**.

**Caching**: results cached in `data/metadata/addressee_cache_validation/` — re-runs are free.

In [ ]:
# ── CHANGE ONLY THESE LINES ───────────────────────────────────────────────────
MODEL        = "claude-haiku-4-5-20251001"  # or claude-sonnet-4-6
ANTHROPIC_API_KEY = ""   # paste key here, or set in .env as ANTHROPIC_API_KEY

DRY_RUN      = False   # True = print first prompt, zero API calls
TEST_ONE_ROW = False  # True = run exactly ONE row then stop

FILM_FILTER  = None   # None = all remaining; or e.g. "frozen_2013"
GROQ_TPD_LIMIT = 100_000
# ──────────────────────────────────────────────────────────────────────────────

In [39]:
import os, json, time, re
from pathlib import Path
import pandas as pd

def _find_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if c.name == "CLEAN" and (c/"data").exists():
            return c
        child = c / "CLEAN"
        if child.is_dir() and (child/"data").exists():
            return child
    raise FileNotFoundError("Cannot find CLEAN root")

CLEAN_ROOT = _find_root()
DATA_FILE  = CLEAN_ROOT / "data" / "03_validation_dataset" / "addressee_validation_results" / "data_for_prediction.xlsx"
CHARS_FILE = CLEAN_ROOT / "data" / "00_corpus" / "characters.csv"
OUT_CSV    = CLEAN_ROOT / "data" / "03_validation_dataset" / "addressee_validation_input" / "validation_with_predictions.csv"
CACHE_DIR  = CLEAN_ROOT / "data" / "03_validation_dataset" / "addressee_cache_validation"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"CLEAN root : {CLEAN_ROOT}")
print(f"Input      : {DATA_FILE}")
print(f"Cache dir  : {CACHE_DIR}")

CLEAN root : /Users/sofapirogova/Documents/final_thesis_1106/final_master_thesis/CLEAN
Input      : /Users/sofapirogova/Documents/final_thesis_1106/final_master_thesis/CLEAN/data/data_for_prediction.xlsx
Cache dir  : /Users/sofapirogova/Documents/final_thesis_1106/final_master_thesis/CLEAN/data/metadata/addressee_cache_validation


In [40]:
import os
import anthropic as _sdk
from dotenv import load_dotenv

load_dotenv(CLEAN_ROOT / ".env")

api_key = ANTHROPIC_API_KEY or os.environ.get("ANTHROPIC_API_KEY", "")
if not api_key and not DRY_RUN:
    raise EnvironmentError("Set ANTHROPIC_API_KEY in parameters or in CLEAN/.env")

client = _sdk.Anthropic(api_key=api_key) if not DRY_RUN else None
print(f"Anthropic client ready. Model: {MODEL}" if not DRY_RUN else "DRY_RUN=True — skipping client setup.")

Anthropic client ready. Model: claude-haiku-4-5-20251001


In [41]:
SYSTEM_MESSAGE = (
    "You are an expert dialogue analyst annotating addressees in Disney/Pixar animated film "
    "screenplays for an academic social network analysis study. "
    "Your annotations will be used to build directed speech networks between characters. "
    "Accuracy and consistency are critical. "
    "Your output must be valid JSON only. No explanation, no markdown, no commentary."
)

VALID_TYPES = {"individual", "group", "monologue", "non_human", "unclear"}

FEW_SHOT_EXAMPLES = """
## EXAMPLES (study these before annotating)

EXAMPLE 1 — Direct address by name (high confidence)
Scene: Gaston and Belle are talking outside her cottage.
  u001 | Gaston: Belle, I brought you a present.
  u002 | Belle: Antlers! How thoughtful! Thank you, but I couldn't.
Target: u001 | Gaston: "Belle, I brought you a present."
Reasoning: Gaston calls Belle by name and hands her something directly.
Answer: {"addressee": "beautyandthebeast_1991_belle", "addressee_type": "individual", "confidence": "high"}

EXAMPLE 2 — Response-chain inference (no explicit name)
Scene: Woody and Buzz are arguing in Andy's room.
  u001 | Buzz: I am Buzz Lightyear. I come in peace.
  u002 | Woody: Oh, right. Hey look, the word of a toy that thinks he's a real space ranger.
Target: u002 | Woody: "Oh, right. Hey look, the word of a toy..."
Reasoning: Woody is directly responding to Buzz's previous utterance; Buzz is the only other character present.
Answer: {"addressee": "toy_story_1995_buzz", "addressee_type": "individual", "confidence": "high"}

EXAMPLE 3 — Group address
Scene: Mirabel addresses her whole family gathered in the dining room.
  u001 | Mirabel: I know I'm not the one with a gift, but I love this family. All of you.
Target: u001 | Mirabel: "I know I'm not the one with a gift, but I love this family. All of you."
Reasoning: "All of you" explicitly signals the entire family is the audience.
Answer: {"addressee": "group", "addressee_type": "group", "confidence": "high"}

EXAMPLE 4 — Monologue
Scene: Joy alone in the memory dump, no other characters present.
  u001 | Joy: Come on, Joy. There has to be a way. Think, think, think...
Target: u001 | Joy: "Come on, Joy. Think, think, think..."
Reasoning: Joy is physically alone and talking to herself.
Answer: {"addressee": "monologue", "addressee_type": "monologue", "confidence": "high"}

EXAMPLE 5 — Implicit on-stage inference
Scene: Only Sulley and Mike are in the locker room.
  u001 | Sulley: I don't know about this, buddy.
Target: u001 | Sulley: "I don't know about this, buddy."
Reasoning: "Buddy" is informal; Mike is the only other character on-stage.
Answer: {"addressee": "monsters_inc_2001_mike", "addressee_type": "individual", "confidence": "high"}
"""

PRIORITY_RULES = """
## PRIORITY RULES (apply in order, stop at first match)
1. PARENTHETICAL: stage direction like (to Belle) → follow it
2. DIRECT NAME/PRONOUN: speaker uses addressee's name or "you" in a two-person scene
3. RESPONSE CHAIN: this utterance directly answers the immediately preceding speaker
4. SCENE PRESENCE: infer from who is physically present on-stage
5. If none apply → "unclear"
"""


def build_prompt(row, char_map: dict) -> str:
    char_lines = [f"  {cid}  ({name})" for cid, name in sorted(char_map.items())]
    char_lines += [
        "  group      -> addressed to multiple characters simultaneously",
        "  monologue  -> spoken aloud to oneself, no intended listener",
        "  non_human  -> addressed to an animal, object, or magical entity",
        "  unclear    -> genuinely impossible to determine",
    ]
    uid_val = row["utterance_id"]
    return (
        FEW_SHOT_EXAMPLES
        + "\n" + PRIORITY_RULES
        + "\n## NOW ANNOTATE\n"
        + f"\nFILM: {row['film_id']}"
        + f"\n\nSCENE CONTEXT:\n{row['scene_text']}"
        + "\n\nTARGET UTTERANCE:\n"
        + f"  ID      : {row['utterance_id']}\n"
        + f"  Speaker : {row['canonical_name']} ({row['speaker']})\n"
        + f"  Text    : {row['text']}\n"
        + "\nSTEP-BY-STEP (follow priority rules):\n"
        + "Step 1 — parenthetical cue?\n"
        + "Step 2 — direct name/pronoun?\n"
        + "Step 3 — response to previous speaker?\n"
        + "Step 4 — who is on-stage?\n"
        + "\nVALID ADDRESSEE VALUES (exact string, case-sensitive):\n"
        + "\n".join(char_lines)
        + "\n\nReturn ONLY this JSON:\n"
        + '{"utterance_id": "' + uid_val + '", '
        + '"step1": "<found or none>", '
        + '"step2": "<found or none>", '
        + '"step3": "<found or none>", '
        + '"step4": "<inference>", '
        + '"addressee": "<exact value>", '
        + '"addressee_type": "<individual|group|monologue|non_human|unclear>", '
        + '"confidence": "<high|medium|low>"}'
    )

def call_llm(prompt: str, max_retries: int = 6):
    """Returns (raw_text, total_tokens)."""
    last_err = None
    for attempt in range(max_retries):
        try:
            r = client.messages.create(
                model=MODEL, max_tokens=1024, temperature=0,
                system=SYSTEM_MESSAGE,
                messages=[{"role": "user", "content": prompt}]
            )
            total = r.usage.input_tokens + r.usage.output_tokens
            return r.content[0].text, total
        except Exception as e:
            last_err = e
            if "529" in str(e) or "overloaded" in str(e).lower() or "429" in str(e):
                wait = 15 * (2 ** attempt)
                print(f"  API busy — waiting {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise last_err


def parse_response(raw: str, char_map: dict):
    valid_addressees = set(char_map.keys()) | SPECIAL
    try:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        data = json.loads(m.group(0) if m else raw)
    except Exception:
        return "unclear", "unclear", "low"
    addressee      = str(data.get("addressee", "unclear")).strip()
    addressee_type = str(data.get("addressee_type", "unclear")).strip()
    confidence     = str(data.get("confidence", "low")).strip()
    if addressee not in valid_addressees:
        print(f"  ⚠ Invalid addressee '{addressee}' → unclear")
        addressee = "unclear"
    if addressee_type not in VALID_TYPES:
        addressee_type = "unclear"
    return addressee, addressee_type, confidence


def cache_path(uid): return CACHE_DIR / f"{uid}.json"
def load_cache(uid):
    p = cache_path(uid)
    return json.loads(p.read_text()) if p.exists() else None
def save_cache(uid, prompt, raw, addressee, atype, conf, model=""):
    cache_path(uid).write_text(json.dumps({
        "utterance_id": uid, "prompt": prompt, "raw_response": raw,
        "addressee": addressee, "addressee_type": atype,
        "confidence": conf, "model_used": model,
    }, ensure_ascii=False, indent=2))

In [42]:
# Reload df fresh from file to get current state
df    = pd.read_excel(DATA_FILE, dtype=str).fillna("")
chars = pd.read_csv(CHARS_FILE, dtype=str).fillna("")

# Rebuild film_chars
film_chars = {}
for film_id, g in chars.groupby("film_id"):
    named = g[g["is_named"].str.lower().isin(["true","1"]) & ~g["canonical_name"].str.contains("/", na=False)]
    film_chars[film_id] = dict(zip(named["character_id"].str.strip(), named["canonical_name"].str.strip()))

# Rows that need prediction: empty OR previously errored
needs = df[df["addressee_llm"].str.strip().isin(["", "error"])].copy()
if FILM_FILTER:
    needs = needs[needs["film_id"] == FILM_FILTER].copy()
    print(f"Film filter: {FILM_FILTER}")

n_cached = sum(1 for uid in needs["utterance_id"] if load_cache(uid) and load_cache(uid).get("addressee","") not in ["","error"])
n_calls  = len(needs) - n_cached
est_tokens = n_calls * 2200

print(f"Rows to process  : {len(needs)}")
print(f"Already cached   : {n_cached}")
print(f"API calls needed : {n_calls}")
print(f"Est. tokens      : ~{est_tokens:,}")
print(f"Est. cost (Haiku): ~${est_tokens * 0.000001 * 1.8:.3f}  (Sonnet: ~${est_tokens * 0.000001 * 9:.3f})")
print(f"Temperature      : 0 (deterministic)")

if DRY_RUN:
    row = needs.iloc[0]
    char_map = film_chars.get(row["film_id"], {})
    print("\n" + "="*70)
    print("SYSTEM:", SYSTEM_MESSAGE)
    print("\nUSER PROMPT:")
    print(build_prompt(row, char_map))
    print("="*70)
    print("\nSet DRY_RUN=False + TEST_ONE_ROW=True to run one real call.")
else:
    tokens_used = 0
    for i, (idx, row) in enumerate(needs.iterrows()):
        uid      = row["utterance_id"]
        char_map = film_chars.get(row["film_id"], {})

        cached = load_cache(uid)
        if cached and cached.get("addressee","") not in ["","error"]:
            df.at[idx, "addressee_llm"]      = cached["addressee"]
            df.at[idx, "addressee_type_llm"] = cached["addressee_type"]
            df.at[idx, "model_used"]         = cached.get("model_used","")
            continue

        if TEST_ONE_ROW and i > 0:
            print("TEST_ONE_ROW=True — stopped. Check output, then set False.")
            break

        prompt = build_prompt(row, char_map)
        try:
            raw, tok = call_llm(prompt)
            tokens_used += tok
            addressee, atype, conf = parse_response(raw, char_map)
            save_cache(uid, prompt, raw, addressee, atype, conf, f"anthropic/{MODEL}")
            df.at[idx, "addressee_llm"]      = addressee
            df.at[idx, "addressee_type_llm"] = atype
            df.at[idx, "model_used"]         = f"anthropic/{MODEL}"
            n_left = n_calls - i - 1
            print(f"[{i+1:>3}/{n_calls}] {uid[-45:]}  →  {addressee} / {atype} [{conf}]  ({n_left} left)  [tokens: {tokens_used:,}]")
            if i < len(needs) - 1:
                time.sleep(0.2)
        except Exception as e:
            print(f"  ERROR {uid}: {e}")
            df.at[idx, "addressee_llm"]      = "error"
            df.at[idx, "addressee_type_llm"] = "error"

    df.to_csv(OUT_CSV, index=False)
    df.to_excel(DATA_FILE, index=False)
    print(f"\nTokens used: {tokens_used:,}  Cost: ~${tokens_used*0.000001*1.8:.4f}")
    print(f"Saved to {OUT_CSV}")

Rows to process  : 183
Already cached   : 0
API calls needed : 183
Est. tokens      : ~402,600
Est. cost (Haiku): ~$0.725  (Sonnet: ~$3.623)
Temperature      : 0 (deterministic)
[  1/183] findingnemo_s017_u1111  →  findingnemo_nemo / individual [high]  (182 left)  [tokens: 2,332]
[  2/183] findingnemo_s019_u1208  →  findingnemo_darla / individual [high]  (181 left)  [tokens: 4,987]
[  3/183] findingnemo_s023_u1313  →  findingnemo_marlin / individual [high]  (180 left)  [tokens: 7,647]
[  4/183] findingnemo_s025_u1405  →  findingnemo_dentist / individual [high]  (179 left)  [tokens: 9,816]
[  5/183] frozen_2013_s013_u0076  →  non_human / non_human [high]  (178 left)  [tokens: 11,479]
[  6/183] frozen_2013_s017_u0150  →  frozen_2013_elsa / individual [high]  (177 left)  [tokens: 13,608]
[  7/183] frozen_2013_s018_u0217  →  frozen_2013_anna / individual [high]  (176 left)  [tokens: 15,566]
[  8/183] frozen_2013_s021_u0283  →  frozen_2013_anna / individual [high]  (175 left)  [tokens: 17,4

In [43]:
# Summary
filled = df[df["addressee_llm"].str.strip().isin(set(df["addressee_llm"].unique()) - {"","error"})]
print(f"Filled: {len(filled)}/{len(df)} rows\n")
print("addressee_type_llm distribution:")
print(filled["addressee_type_llm"].value_counts().to_string())
print("\nPer film:")
print(filled.groupby("film_id")["addressee_llm"].count().to_string())

Filled: 270/270 rows

addressee_type_llm distribution:
addressee_type_llm
individual    208
group          45
monologue       9
non_human       6
unclear         2

Per film:
film_id
beautyandthebeast_1991           15
cars2                            15
coco_2017                        15
elemental_2023                   15
encanto_2021                     15
findingnemo                      15
frozen_2013                      15
incredibles_2_2018               15
inside_out_2015                  15
monsters_inc_2001                15
mulan_1998                       15
onward_2020                      15
raya_and_the_last_dragon_2021    15
soul_2020                        15
toy_story_1995                   15
toy_story_3_2010                 15
up                               15
zootopia_2016                    15
